In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import scipy.stats as stats

# ==========================================
# 1. CONFIGURATION & SEED
# ==========================================
class Config:
    SEED = 42
    NUM_FACILITIES = 100 # Start small for testing, scale to 5000+ later
    START_DATE = pd.to_datetime('2023-01-01')
    END_DATE = pd.to_datetime('2023-12-31')
    
    # Medicine definitions
    MEDICINES = ['PARACETAMOL', 'RIF_INH', 'EPINEPHRINE']
    
    @classmethod
    def get_rng(cls):
        """Returns a reproducible numpy random generator"""
        return np.random.default_rng(cls.SEED)

# ==========================================
# 2. GENERATE FACILITIES (Static Data)
# ==========================================
def generate_facilities(config):
    rng = config.get_rng()
    n = config.NUM_FACILITIES
    
    # Generate Base IDs
    facility_ids = [f"FAC-{str(i).zfill(4)}" for i in range(1, n + 1)]
    
    # 60% Rural, 40% Urban (Bernoulli/Binomial distribution)
    is_rural = rng.binomial(1, 0.6, size=n)
    region_type = np.where(is_rural == 1, 'Rural', 'Urban')
    facility_type = np.where(is_rural == 1, 'BHS', 'RHU')
    
    # Latent Management Quality (Beta distribution: heavily skewed towards good, but with a left tail)
    # 0 = terrible, 1 = perfect
    latent_quality = rng.beta(a=5, b=2, size=n)
    
    # Population Served (Truncated Normal Distribution based on Facility Type)
    # BHS: mean 5000, RHU: mean 22000
    bhs_pop = rng.normal(loc=5000, scale=800, size=n)
    rhu_pop = rng.normal(loc=22000, scale=4000, size=n)
    
    population = np.where(facility_type == 'BHS', bhs_pop, rhu_pop)
    population = np.clip(population, 2500, 35000).astype(int) # Ensure realistic bounds
    
    # Storage Capacity (Highly correlated with population and facility type)
    storage = (population * 0.5 * rng.uniform(0.8, 1.2, size=n)).astype(int)
    
    # Assemble Dataframe
    df_facilities = pd.DataFrame({
        'facility_id': facility_ids,
        'facility_type': facility_type,
        'region_type': region_type,
        'population_served': population,
        'storage_capacity_liters': storage,
        'reorder_interval_days': 30, # Standard push cycle
        'latent_management_quality': latent_quality # OMITTED VARIABLE (Keep for now)
    })
    
    return df_facilities

# Run and inspect
df_facilities = generate_facilities(Config)
display(df_facilities.head())
# display(df_facilities['latent_management_quality'].hist()) # Optional: visualize the distribution!

In [ ]:
import itertools

# ==========================================
# 3. GENERATE OUTBREAKS & CONSUMPTION
# ==========================================
def generate_consumption(config, df_facilities):
    rng = config.get_rng()
    df_grid = df_grid.merge(
        df_facilities[['facility_id', 'region_type', 'population_served', 'latent_management_quality']], 
        on='facility_id', 
        how='left'
    )
    
    # 1. Create the Time-Series Grid (Cartesian Product)
    date_range = pd.date_range(start=config.START_DATE, end=config.END_DATE)
    
    # Create all combinations of (Facility, Date)
    grid = list(itertools.product(df_facilities['facility_id'], date_range))
    df_grid = pd.DataFrame(grid, columns=['facility_id', 'date'])
    
    # Merge in facility details we need for the math
    df_grid = df_grid.merge(
        df_facilities[['facility_id', 'region_type', 'population_served']], 
        on='facility_id', 
        how='left'
    )
    
    # 2. Generate the Latent Outbreak Intensity (Omitted Variable)
    # We model a sine wave peaking around day 240 (August/September - rainy season)
    df_grid['day_of_year'] = df_grid['date'].dt.dayofyear
    
    # Random amplitude for Rural vs Urban
    A_rural = rng.uniform(0.6, 0.9)
    A_urban = rng.uniform(0.4, 0.7)
    df_grid['amplitude'] = np.where(df_grid['region_type'] == 'Rural', A_rural, A_urban)
    
    # The seasonal curve + random daily noise
    seasonality = np.sin(2 * np.pi * (df_grid['day_of_year'] - 150) / 365)
    noise = rng.normal(0, 0.05, size=len(df_grid))
    
    df_grid['latent_outbreak_intensity'] = np.clip((seasonality * df_grid['amplitude']) + noise, 0, 1)
    
    # 3. Daily Patient Visits
    lambda_visits = df_grid['population_served'] * (0.003 + 0.005 * df_grid['latent_outbreak_intensity'])
    df_grid['patient_consultations'] = rng.poisson(lambda_visits)
    
    # 4. Paracetamol (Spiky, outbreak-driven)
    lambda_para = df_grid['patient_consultations'] * (1.8 + 3.2 * df_grid['latent_outbreak_intensity'])
    df_grid['PARACETAMOL_dispensed'] = rng.poisson(lambda_para)
    
    # 5. TB Meds (Steady, but adherence drops if facility management is poor)
    # Adherence is between 50% (poorly managed) and 100% (perfectly managed)
    adherence = 0.5 + (0.5 * df_grid['latent_management_quality'])
    lambda_tb = df_grid['population_served'] * 0.0004 * adherence
    df_grid['RIF_INH_dispensed'] = rng.poisson(lambda_tb)
    
    # 6. Epinephrine (Rare, acute emergency)
    lambda_epi = np.maximum(0.02, df_grid['population_served'] * 0.00002)
    df_grid['EPINEPHRINE_dispensed'] = rng.poisson(lambda_epi)
    
    # 7. Reshape (Melt) to Long Format
    value_vars = ['PARACETAMOL_dispensed', 'RIF_INH_dispensed', 'EPINEPHRINE_dispensed']
    df_consumption = pd.melt(
        df_grid, 
        id_vars=['facility_id', 'date', 'patient_consultations', 'latent_outbreak_intensity'],
        value_vars=value_vars,
        var_name='medicine_code', 
        value_name='quantity_dispensed'
    )
    
    # Clean up the medicine names
    df_consumption['medicine_code'] = df_consumption['medicine_code'].str.replace('_dispensed', '')
    
    # Add primary key
    df_consumption['consumption_id'] = [f"CON-{str(i).zfill(7)}" for i in range(1, len(df_consumption) + 1)]
    
    return df_consumption

# RUN IT
df_consumption = generate_consumption(Config, df_facilities)
# print(df_consumption.head())